# 07 - Unimodularity and Total Dual Integrality

Integer programs are difficult partly because an LP relaxation can have fractional vertices. This notebook studies structural certificates that rule out that behavior. **Total unimodularity** is a property of a constraint matrix; **total dual integrality** is a property of a particular inequality system. Both can certify that an LP formulation already describes an integral polyhedron.

## Learning objectives

By the end of this notebook, you should be able to:

1. distinguish unimodular and totally unimodular matrices;
2. explain why a totally unimodular matrix with an integral right-hand side defines an integral polyhedron;
3. recognize network matrices as an important source of total unimodularity;
4. distinguish primal integrality, dual integrality, and total dual integrality;
5. use Hermite normal form and the integral Farkas lemma to reason about lattice equations; and
6. explain how Hilbert bases characterize TDI systems.

## Prerequisites

We use vertices, faces, and H-representations from Notebook 04, integral polyhedra from Notebook 05, and basic LP duality. All matrices and vectors are real unless a different domain is shown explicitly.


## Unimodular and totally unimodular matrices

Let $A\in\mathbb{Z}^{m\times n}$ have full row rank, with $m\le n$. It is **unimodular** if every nonsingular $m\times m$ submatrix has determinant $+1$ or $-1$. Equivalently, every maximal square minor belongs to $\{-1,0,1\}$.

The matrix $A$ is **totally unimodular (TU)** if **every** square submatrix has determinant in $\{-1,0,1\}$, regardless of its size.

Thus total unimodularity is stronger:

$$
A\text{ is TU and has full row rank}
\quad\Longrightarrow\quad
A\text{ is unimodular}.
$$

The converse fails. In particular, every entry of a TU matrix must be $-1$, $0$, or $1$, because each entry is a $1\times1$ minor.


In [ ]:
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import linprog
from sympy import Matrix, Rational
from sympy.matrices.normalforms import hermite_normal_form


### Exact minor checks

Floating-point determinants are unsuitable for a definition involving exact integers. The helper below uses SymPy's exact arithmetic and is intended only for small educational examples. A matrix has

$$
\sum_{k=1}^{\min(m,n)}\binom{m}{k}\binom{n}{k}
$$

nonempty square minors, so exhaustive enumeration grows rapidly.


In [ ]:
def minor_determinants(matrix):
    # Return (order, row indices, column indices, determinant) for every square minor.
    matrix = Matrix(matrix)
    records = []
    for order in range(1, min(matrix.shape) + 1):
        for rows in combinations(range(matrix.rows), order):
            for columns in combinations(range(matrix.cols), order):
                determinant = matrix.extract(rows, columns).det()
                records.append((order, rows, columns, determinant))
    return records


def is_totally_unimodular(matrix):
    # Check total unimodularity by exhaustive exact minor enumeration.
    return all(record[3] in {-1, 0, 1} for record in minor_determinants(matrix))


Consider the square matrix

$$
A=
\begin{pmatrix}
2&3&2\\
4&2&3\\
9&6&7
\end{pmatrix}.
$$

Its only maximal minor is its determinant, which equals $1$. It is therefore unimodular. It is not TU: the entry $a_{11}=2$ is already a forbidden $1\times1$ minor.


In [ ]:
A_unimodular = Matrix([[2, 3, 2], [4, 2, 3], [9, 6, 7]])

print("det(A) =", A_unimodular.det())
print("Unimodular:", abs(A_unimodular.det()) == 1)
print("Totally unimodular:", is_totally_unimodular(A_unimodular))


## Why determinants control integrality

Suppose a vertex $x^*$ of

$$
P=\{x\in\mathbb{R}^n\mid Ax\le b\}
$$

is determined by $n$ linearly independent tight inequalities. Let $B$ be their coefficient matrix and $b_B$ the corresponding right-hand side. Then

$$
Bx^*=b_B,
\qquad
x^*=B^{-1}b_B.
$$

Cramer's rule writes each coordinate as a ratio of integer determinants. If $A$ is TU, then a nonsingular square submatrix $B$ has $\det(B)=\pm1$. Consequently $B^{-1}$ is integral, and $b_B\in\mathbb{Z}^n$ implies $x^*\in\mathbb{Z}^n$.

This gives the fundamental result:

> **TU integrality theorem.** If $A$ is totally unimodular and $b$ is integral, then $\{x\mid Ax\le b\}$ is an integral polyhedron.

Equality constraints can be replaced by opposite inequalities, and variable bounds use rows of $I$ and $-I$. Total unimodularity is preserved by transposition, row or column permutations, sign changes, duplication, and adjoining unit rows or columns. Hence TU also gives integrality for common forms such as

$$
\{x\ge0\mid Ax=b\}
\quad\text{and}\quad
\{x\mid Ax=b,\ 0\le x\le u\}
$$

when all right-hand sides are integral.

The assumption $b\in\mathbb{Z}^m$ is essential. The TU system $x\le\tfrac12$, $-x\le0$ has the fractional vertex $x=\tfrac12$.


## Network matrices: a reusable TU structure

For a directed graph, its **node–arc incidence matrix** has one column per arc. The column for an arc $u\to v$ contains $-1$ in row $u$, $+1$ in row $v$, and zeros elsewhere. Every directed node–arc incidence matrix is TU.

The full incidence matrix has dependent rows because each column sums to zero. Removing one node row produces a full-row-rank matrix without changing the flow-conservation information when total supply equals total demand.


In [ ]:
nodes = ["s", "a", "b", "t"]
arcs = [("s", "a"), ("s", "b"), ("a", "b"), ("a", "t"), ("b", "t")]

incidence = np.zeros((len(nodes), len(arcs)), dtype=int)
for arc_index, (tail, head) in enumerate(arcs):
    incidence[nodes.index(tail), arc_index] = -1
    incidence[nodes.index(head), arc_index] = 1

reduced_incidence = incidence[:-1]
print(reduced_incidence)
print("Totally unimodular:", is_totally_unimodular(reduced_incidence))


Consider a minimum-cost flow of three units from $s$ to $t$. Flow conservation is $Dx=d$, where $D$ is the incidence matrix and $d=(-3,0,0,3)^\top$. Integral capacities add $0\le x\le u$. The augmented constraint matrix remains TU, so every vertex of the feasible flow polytope is integral. An LP solver can therefore solve this integer-flow model without explicit integrality restrictions.


In [ ]:
cost = np.array([2, 4, 1, 4, 1], dtype=float)
capacity = np.array([2, 2, 1, 2, 2], dtype=float)
balance = np.array([-3, 0, 0, 3], dtype=float)

flow_result = linprog(
    cost,
    A_eq=reduced_incidence,
    b_eq=balance[:-1],
    bounds=list(zip(np.zeros(len(arcs)), capacity)),
    method="highs",
)

flow = np.rint(flow_result.x).astype(int)
for arc, value in zip(arcs, flow):
    print(f"{arc[0]} -> {arc[1]}: {value}")
print("Minimum cost:", flow_result.fun)
print("Maximum distance from an integer:", np.max(np.abs(flow_result.x - flow)))


Total unimodularity is a property of the matrix, not of one chosen right-hand side or objective. It provides a uniform guarantee over **every** integral right-hand side. Important TU families include directed incidence matrices, matrices of bipartite matching formulations, and matrices with the consecutive-ones property.

Not every $\{-1,0,1\}$ matrix is TU. For example,

$$
\begin{pmatrix}
1&1&0\\
1&0&1\\
0&1&1
\end{pmatrix}
$$

has determinant $-2$. Exhaustive minor enumeration can disprove TU on small matrices, but structural theorems are preferable for realistic models.


In [ ]:
not_tu = Matrix([[1, 1, 0], [1, 0, 1], [0, 1, 1]])
print("determinant =", not_tu.det())
print("Totally unimodular:", is_totally_unimodular(not_tu))


## From LP duality to total dual integrality

For the primal maximization problem

$$
\max\{c^\top x\mid Ax\le b\},
$$

with unrestricted $x$, the dual is

$$
\min\{b^\top y\mid A^\top y=c,\ y\ge0\}.
$$

A rational system $Ax\le b$ is **totally dual integral (TDI)** if, for every $c\in\mathbb{Z}^n$ for which the dual has an optimum, the dual has an optimal solution $y^*\in\mathbb{Z}_{\ge0}^m$.

The quantifiers matter:

- **primal integrality** concerns the geometry of $P=\{x\mid Ax\le b\}$;
- **dual integrality** for one LP concerns one objective $c$; and
- **total dual integrality** requires an integral dual optimum for every integral objective whenever an optimum exists.

TDI is a property of the displayed system, not just of the polyhedron it describes.


### The main TU and TDI implications

Two central results connect the concepts:

1. If $A$ is TU, then $Ax\le b$ is TDI for every rational $b$.
2. If $Ax\le b$ is TDI and $b$ is integral, then $P=\{x\mid Ax\le b\}$ is integral.

For the first result, the dual feasible region $\{y\ge0\mid A^\top y=c\}$ has a TU constraint matrix. When $c$ is integral, its vertices are integral, so an attained dual optimum can be chosen integral.

For the second result, let $c\in\mathbb{Z}^n$ and assume the primal optimum is finite. Strong duality and TDI give

$$
\max\{c^\top x\mid Ax\le b\}
=b^\top y^*
\in\mathbb{Z}
$$

for an integral optimal dual vector $y^*$. The resulting integer support-function values imply that the rational polyhedron is integral.

Together,

$$
A\text{ TU},\ b\in\mathbb{Z}^m
\quad\Longrightarrow\quad
Ax\le b\text{ is TDI}
\quad\Longrightarrow\quad
P(A,b)\text{ is integral}.
$$

The converses do not hold in general.


### The description matters

The interval $P=[0,1]$ has both descriptions

$$
\begin{aligned}
x&\le1, & -x&\le0,\\
2x&\le2, & -x&\le0.
\end{aligned}
$$

The first coefficient matrix is TU, so the first system is TDI. For the second system, choose $c=1$. Its dual is

$$
\min\{2y_1\mid 2y_1-y_2=1,\ y\ge0\}.
$$

The unique optimal choice for $y_1$ is $y_1=\tfrac12$ and then $y_2=0$. The value is $1$, but no integral dual optimum exists. Thus the second description is not TDI even though it defines exactly the same integral polyhedron.

Likewise, an integral polyhedron need not be presented by a TU matrix. TDI is more flexible than TU because it can certify integral polyhedra through the particular interaction of $A$ and $b$.


## A lattice interlude: Hermite normal form

The next ideas explain algebra behind integral feasibility and TDI characterizations. Let $A\in\mathbb{Z}^{m\times n}$ have full row rank. Its columns generate the lattice

$$
\mathcal{L}(A)=\{Az\mid z\in\mathbb{Z}^n\}.
$$

A unimodular matrix $U\in\mathbb{Z}^{n\times n}$ satisfies $\det(U)=\pm1$, so both $U$ and $U^{-1}$ are integral. Therefore

$$
\{AUz\mid z\in\mathbb{Z}^n\}=\mathcal{L}(A).
$$

Integer column operations—swapping columns, changing a column's sign, or adding an integer multiple of one column to another—amount to multiplication by such a $U$ and preserve the column lattice.

These operations can transform $A$ to a **column Hermite normal form** $[H\;0]$, subject to a chosen convention. The triangular matrix $H$ gives a canonical basis of the column lattice. For rational data, denominators are first cleared to obtain an equivalent integer lattice problem.


In [ ]:
A_lattice = Matrix([[2, 0, 1], [0, 2, 1]])
H = hermite_normal_form(A_lattice)

print("A =")
print(A_lattice)
print("Column Hermite normal form H =")
print(H)


Here the columns generate exactly the integer vectors whose two coordinates have the same parity. Indeed,

$$
H=
\begin{pmatrix}
2&1\\
0&1
\end{pmatrix},
\qquad
Hz=
\begin{pmatrix}
2z_1+z_2\\z_2
\end{pmatrix}.
$$

Thus $Ax=b$ has an integer solution precisely when $H^{-1}b$ is integral. For example, $(3,1)^\top$ belongs to the lattice, whereas $(1,0)^\top$ does not.


In [ ]:
for right_hand_side in (Matrix([3, 1]), Matrix([1, 0])):
    coordinates = H.inv() * right_hand_side
    is_integral = all(value.q == 1 for value in coordinates)
    print(
        f"b = {tuple(right_hand_side)}: H^(-1)b = {tuple(coordinates)}, "
        f"integer-feasible = {is_integral}"
    )


## The integral Farkas lemma

Ordinary Farkas lemmas certify real feasibility of inequalities. For unrestricted integer solutions of a rational equality system, the relevant alternative is:

> **Integral Farkas lemma.** For $A\in\mathbb{Q}^{m\times n}$ and $b\in\mathbb{Q}^m$, exactly one of the following holds:
>
> 1. there is an $x\in\mathbb{Z}^n$ such that $Ax=b$;
> 2. there is a $y\in\mathbb{Q}^m$ such that $y^\top A\in\mathbb{Z}^n$ but $y^\top b\notin\mathbb{Z}$.

Why can the two statements not both hold? If $Ax=b$ for an integer $x$ and $y^\top A$ is integral, then

$$
y^\top b=y^\top Ax=(y^\top A)x\in\mathbb{Z},
$$

contradicting the second statement.

For the infeasible lattice right-hand side $b=(1,0)^\top$ above, choose $y=(\tfrac12,-\tfrac12)^\top$. Then

$$
y^\top A=(1,-1,0)\in\mathbb{Z}^3,
\qquad y^\top b=\tfrac12\notin\mathbb{Z}.
$$

This lemma addresses unrestricted equations $Ax=b$ with $x\in\mathbb{Z}^n$. Adding inequalities, nonnegativity, or binary restrictions leads to different and generally harder problems.


In [ ]:
y_certificate = Matrix([[Rational(1, 2), Rational(-1, 2)]])
b_infeasible = Matrix([1, 0])

print("y^T A =", y_certificate * A_lattice)
print("y^T b =", (y_certificate * b_infeasible)[0])


## Hilbert bases

Let $C\subseteq\mathbb{R}^n$ be a rational polyhedral cone. A finite set

$$
H=\{h^1,\ldots,h^t\}\subseteq C\cap\mathbb{Z}^n
$$

is a **Hilbert basis** of $C$ if every lattice point $z\in C\cap\mathbb{Z}^n$ can be written as

$$
z=\sum_{i=1}^t\lambda_i h^i,
\qquad
\lambda_i\in\mathbb{Z}_{\ge0}.
$$

Real conic generation is not enough: nonnegative **integer** coefficients must generate every lattice point in the cone.

Consider

$$
C=\operatorname{cone}\left\{
\begin{pmatrix}1\\0\end{pmatrix},
\begin{pmatrix}1\\2\end{pmatrix}
\right\}.
$$

The primitive extreme-ray generators do not generate $(1,1)^\top$ with nonnegative integer coefficients. The missing vector must be added, and

$$
H=\left\{
\begin{pmatrix}1\\0\end{pmatrix},
\begin{pmatrix}1\\1\end{pmatrix},
\begin{pmatrix}1\\2\end{pmatrix}
\right\}
$$

is a Hilbert basis.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

lattice_points = [(x, y) for x in range(6) for y in range(11) if 0 <= y <= 2 * x]
point_x, point_y = zip(*lattice_points)
ax.scatter(point_x, point_y, color="0.65", label=r"$C\cap\mathbb{Z}^2$")

hilbert_basis = np.array([[1, 0], [1, 1], [1, 2]])
for index, vector in enumerate(hilbert_basis):
    ax.arrow(
        0,
        0,
        vector[0],
        vector[1],
        width=0.025,
        length_includes_head=True,
        color="tab:blue",
        label="Hilbert-basis vectors" if index == 0 else None,
    )

ax.plot([0, 5], [0, 0], color="tab:orange", linewidth=2)
ax.plot([0, 5], [0, 10], color="tab:orange", linewidth=2, label="Extreme rays")
ax.set(xlim=(-0.2, 5.3), ylim=(-0.3, 10.5), xlabel=r"$z_1$", ylabel=r"$z_2$")
ax.set_title("A Hilbert basis needs an interior generator")
ax.grid(alpha=0.25)
ax.legend()
plt.show()


To verify the claim, take $(x,y)\in C\cap\mathbb{Z}^2$, so $x\ge0$ and $0\le y\le2x$. Write $y=2q+r$ with $r\in\{0,1\}$. Then

$$
(x,y)=(x-q-r)(1,0)+r(1,1)+q(1,2).
$$

All three coefficients are nonnegative integers. In particular, if $r=1$, the inequality $2q+1\le2x$ gives $q+1\le x$.


## Hilbert bases characterize TDI systems

Let $P=\{x\mid Ax\le b\}\ne\varnothing$, with $A$ integral. For a nonempty face $F$ of $P$, define the inequalities that are tight throughout that face by

$$
I(F)=\{i\mid a_i^\top x=b_i\text{ for every }x\in F\}.
$$

A fundamental characterization states:

$$
\boxed{
Ax\le b\text{ is TDI}
\iff
\{a_i\mid i\in I(F)\}\text{ is a Hilbert basis of }
\operatorname{cone}\{a_i\mid i\in I(F)\}
\text{ for every nonempty face }F.
}
$$

The connection comes from LP optimality. If $F$ is the face maximizing an integral objective $c$, then dual feasibility expresses

$$
c=A^\top y=\sum_i y_i a_i,
\qquad y\ge0.
$$

Complementary slackness permits positive multipliers only on inequalities tight on $F$. TDI asks for integral multipliers; the Hilbert-basis condition says exactly that every integral $c$ in the relevant cone has such a nonnegative integer representation.

This criterion also reveals why TDI can depend on redundant or rescaled inequalities: changing the listed row generators can change their integer generating property even when the feasible set stays fixed.


## What each certificate tells us

The ideas in this notebook form a hierarchy, but they answer different questions:

| Certificate | Object being certified | Integrality conclusion |
|---|---|---|
| $A$ is unimodular | Full-row-rank matrix in standard form | Basic solutions of $Ax=b$, $x\ge0$, are integral for integral $b$ |
| $A$ is TU | Matrix, uniformly over right-hand sides | $P(A,b)$ is integral for every integral $b$ |
| $Ax\le b$ is TDI | One inequality description | Integral dual optima exist for every integral objective |
| TDI and $b$ integral | One inequality description and RHS | $P(A,b)$ is integral |
| Active rows form Hilbert bases | Every face of one description | The system is TDI |

A rational polyhedron is integral if and only if it has some TDI description with integral $A$ and $b$. It need not have an obvious TU description. In modeling practice, recognizing a TU formulation is especially valuable because it is a direct, objective-independent certificate that ordinary LP solves the integer model.


## Check your understanding

1. Why must every entry of a TU matrix belong to $\{-1,0,1\}$?
2. What is the difference between unimodularity and total unimodularity?
3. Where is the assumption that $b$ is integral used in the TU integrality theorem?
4. Why may one row of a node–arc incidence matrix be deleted?
5. Can two systems defining the same polyhedron differ in whether they are TDI?
6. In the Hilbert-basis example, why are the two extreme-ray generators insufficient?
7. How does complementary slackness connect active rows of a face to dual multipliers?

Brief answers: (1) entries are $1\times1$ minors; (2) unimodularity checks maximal minors, while TU checks every square minor; (3) it makes $B^{-1}b_B$ integral because $B^{-1}$ is integral; (4) incidence rows sum to zero, and one conservation equation is redundant when balances sum to zero; (5) yes—TDI is a property of a description; (6) they cannot generate $(1,1)$ with nonnegative integer coefficients; (7) only rows tight on the optimal face need positive dual multipliers.


## Takeaways and further reading

- Total unimodularity turns determinant information into a uniform integrality guarantee.
- Directed node–arc incidence matrices are TU, which explains why many network-flow integer programs can be solved as LPs.
- TDI requires an integral dual optimum for every integral objective whenever a dual optimum exists.
- TU implies TDI, and TDI with an integral right-hand side implies an integral polyhedron.
- TDI belongs to a system, not merely to its feasible set.
- Hermite normal form describes column lattices, while the integral Farkas lemma certifies infeasible lattice equations.
- Hilbert bases translate integral dual multipliers into an integer-generation condition on active constraint rows.

For comprehensive treatments, see Alexander Schrijver, *Theory of Linear and Integer Programming*, Chapters 19 and 22, and Gérard Cornuéjols, *Combinatorial Optimization: Packing and Covering*, Chapter 2. For network formulations, see Ravindra Ahuja, Thomas Magnanti, and James Orlin, *Network Flows*.


## Next

An integral formulation is ideal, but a general LP relaxation is often fractional. The next notebook develops Gomory cuts: valid inequalities derived from tableau rows that remove fractional LP solutions while preserving integer-feasible points.

➡️ [08 - Fractional and Mixed-Integer Gomory Cuts](08-Fractional-and-Mixed-Integer-Gomory-Cuts.ipynb)
